# Data Science Project: Graph & LSH Document Recommendation

This notebook is organized into three main sections:
1. **Graph-based Community Detection**: Using Louvain modularity to cluster articles and retrieval via embeddings.
2. **Locality Sensitive Hashing (LSH)**: Using MinHash and LSH for fast approximate nearest neighbor search.
3. **Evaluation**: Metrics to assess the performance of these methods.

---

# Part 1: Graph Community Detection & Embedding Retrieval

This method constructs a graph where nodes are articles and edges are citations. Communities are detected to group related papers. Retrieval is a two-step process: finding an entry point via embeddings, then expanding search within the community.

## 1.1 Libraries & Imports
Core libraries for graph processing (NetworkX), embeddings (SentenceTransformers), and vector search (Faiss).

In [1]:
import networkx as nx
from networkx.algorithms.community import louvain_communities
from networkx.algorithms.community.quality import modularity
import json
import matplotlib.pyplot as plt
from operator import itemgetter
from typing import Set, Dict, Any, Tuple
import os
import numpy as np

from operator import itemgetter # Utilisé pour trier

In [6]:
from sentence_transformers import SentenceTransformer

# Define output paths variables to ensure consistency
output_emb_file = f'data/embeddings/embeddings_{d_type}.npy'
output_ids_file = f'data/embeddings/doc_ids_{d_type}.json'

# Check if the output files already exist
if os.path.exists(output_emb_file) and os.path.exists(output_ids_file):
    # Files exist: Skip the expensive computation
    print(f"Embeddings and IDs found at '{output_emb_file}'. Skipping computation.")
else:
    # Files do not exist: Proceed with loading data and computing embeddings
    print("Output files not found. Starting data loading and embedding computation...")

    with open(f"data/processed/clean_subdataset_{d_type}.json", 'r', encoding='utf-8') as f:
        data = json.load(f)

    texts = [article["clean_text"] for article in data["articles"]]
    doc_ids = [article["id"] for article in data["articles"]]

    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

    # Call the function
    compute_and_save_embeddings(
        texts,
        doc_ids,
        model_name='all-MiniLM-L6-v2',
        batch_size=8,
        out_emb_path=output_emb_file,
        out_ids_path=output_ids_file,
        overwrite=True
    )
    print("Computation finished and files saved.")

# Loading and verification (Load from the specific paths defined above)
print("-" * 30)
print("Verifying loaded data...")

loaded_emb = np.load(output_emb_file, mmap_mode='r')
with open(output_ids_file, 'r', encoding='utf-8') as f:
    loaded_ids = json.load(f)

print("Embeddings shape:", loaded_emb.shape)
print("Number of doc ids:", len(loaded_ids))
print("Example embedding (first doc) first 10 dims:", loaded_emb[0][:10])

/Users/surprisedcat/DTU/DS/DTU_DS_PROJECT_69/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Embeddings and IDs found at 'data/embeddings/embeddings_waterfilling.npy'. Skipping computation.
------------------------------
Verifying loaded data...
Embeddings shape: (49980, 384)
Number of doc ids: 49980
Example embedding (first doc) first 10 dims: [-0.09529466 -0.05684136  0.06107605  0.07362463  0.04787469 -0.00985583
 -0.06664003  0.01534571 -0.05632101  0.07219958]


## 1.2 Function Definitions
Here we define all core logic for the graph pipeline:
- `create_graph_from_filtered_json`: Loads data and builds the NetworkX graph.
- `community_detection`: Applies Louvain algorithm.
- `compute_and_save_embeddings`: Generates vector representations of text.
- `graph_retrieval_pipeline`: The main hybrid recommendation function.

In [2]:
d_type = "waterfilling"
FILTERED_JSON_PATH = f"data/processed/clean_subdataset_{d_type}.json"


def create_graph_from_filtered_json(
    json_path: str = FILTERED_JSON_PATH,
) -> nx.DiGraph:
    """
    Builds a directed graph (DiGraph) from the filtered JSON file.
    The graph is created in memory and is NOT saved or loaded from disk.
    """

    print("--- Creating graph from filtered JSON (in memory)... ---")
    
    G = nx.DiGraph()
    
    # 1. Load JSON data (Consolidated try/except)
    try:
        with open(json_path, 'r', encoding='utf-8') as json_file:
            data = json.load(json_file)
            
        articles = data.get('articles', [])
        
        # 2. Graph creation
        for article in articles:
            article_id = article.get('id')
            if article_id is not None:
                G.add_node(article_id) 
                for link in article.get('refs', []):
                    G.add_edge(article_id, link)
                    
             
    except FileNotFoundError:
        print(f"Error: Filtered JSON file not found at: {json_path}. Run 'filter_json_and_save' first.")
        return nx.DiGraph()
    except json.JSONDecodeError:
        print(f"Error: Invalid JSON format in file: {json_path}")
        return nx.DiGraph()
    except Exception as e:
        print(f"Error during graph creation: {e}")
        return nx.DiGraph()

    print(f"Graph created: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges.")
    
    return G

In [3]:
def community_detection():
    results = [] 
    G = create_graph_from_filtered_json()


    # Louvain community detection
    communities = louvain_communities(G, seed=42)

    for comm in communities:
        # Identify the node with the highest degree as the representative
        # representative_article = sorted(comm, key=lambda x: G.in_degree(x), reverse=True)[0]
        results.append({
            # 'representative_node' : representative_article,
            # # Convert set to list for JSON serialization
            'community' : sorted(comm, key=lambda x: G.in_degree(x), reverse=True)
        })

    with open(f'data/processed/communities_{d_type}.json', 'w') as outfile:
        # Use indent for better JSON readability
        json.dump(results, outfile, indent=4) 


In [5]:
def compute_and_save_embeddings(texts, doc_ids, model_name='all-mpnet-base-v2',
                                batch_size=64, out_emb_path='embeddings.npy',
                                out_ids_path='doc_ids.json', overwrite=True):
    if os.path.exists(out_emb_path) and not overwrite:
        raise FileExistsError(f"{out_emb_path} already exists. Set overwrite=True to replace.")

    model = SentenceTransformer(model_name)
    n = len(texts)
    emb_dim = model.get_sentence_embedding_dimension()

    emb_memmap = np.lib.format.open_memmap(out_emb_path, mode='w+', dtype='float32', shape=(n, emb_dim))

    for i in tqdm(range(0, n, batch_size), desc="Embedding batches"):
        batch_texts = texts[i:i+batch_size]
        batch_emb = model.encode(batch_texts, show_progress_bar=False, convert_to_numpy=True)
        # normalize rows to unit vectors (for cosine via inner product)
        norms = np.linalg.norm(batch_emb, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        batch_emb = batch_emb / norms
        emb_memmap[i:i+len(batch_emb)] = batch_emb.astype('float32')

    # ensure data flushed to disk
    del emb_memmap

    # save doc ids as JSON
    with open(out_ids_path, 'w', encoding='utf-8') as f:
        json.dump(list(doc_ids), f, ensure_ascii=False)

    print(f"Saved embeddings -> {out_emb_path}")
    print(f"Saved doc ids -> {out_ids_path}")

In [6]:
def build_faiss_index(embeddings_path=f'data/embeddings/embeddings_{d_type}.npy', index_path=f'data/embeddings/faiss_{d_type}.index',
                      index_type='hnsw', ef_construction=200, M=32):
    if os.path.exists(index_path):
        return faiss.read_index(index_path)
    
    emb = np.load(embeddings_path, mmap_mode='r')  # shape (N, d)
    d = emb.shape[1]
    if index_type == 'flat':
        index = faiss.IndexFlatIP(d)  # inner product -> cosine if vectors normalized
        index.add(emb)
    elif index_type == 'hnsw':
        index = faiss.IndexHNSWFlat(d, M)  # M controls connectivity
        index.hnsw.efConstruction = ef_construction
        index.add(emb)
    else:
        raise ValueError('index_type not supported')
    faiss.write_index(index, index_path)
    return index

In [7]:
def retrieve_similar_articles(query, model, embeddings, articles, index, top_n=5, use_ann=False):
    query_emb = model.encode([query], convert_to_numpy=True)
    # normalize
    query_emb = query_emb / np.linalg.norm(query_emb, axis=1, keepdims=True)
    
    if use_ann:
        distances, indices = index.search(query_emb.astype('float32'), top_n)
    else:
        embeddings = np.load(embeddings, mmap_mode='r')
        # Exact search using sklearn
        nbrs = NearestNeighbors(n_neighbors=top_n, metric="cosine").fit(embeddings)
        distances, indices = nbrs.kneighbors(query_emb)
    
    results = pd.DataFrame({
        "article_id": [articles[i] for i in indices[0]],
        "similarity": [1 - d for d in distances[0]]
    })
    return results

In [8]:
def get_representative(member_id, file_path, top_n=1):
    """
    Récupère une liste d'IDs représentatifs de la communauté du member_id.
    Renvoie une liste vide [] si le noeud n'est pas trouvé ou s'il y a une erreur.
    """
    try:
        # Note: Pour la performance, il est préférable de charger ce JSON 
        # une seule fois en dehors de la fonction si le fichier est gros.
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        for group in data:
            community = group.get("community", [])
            if member_id in community:
                # On s'assure de ne pas demander plus d'articles que la communauté n'en contient
                limit = min(top_n, len(community))
                # On retourne une LISTE (slice), et non un élément unique
                return community[:limit]
                
        return []

    except (FileNotFoundError, json.JSONDecodeError):
        print("[Get representative]: json file not found or invalid")
        return []

In [9]:
def graph_retrieval_pipeline(query_text, n, **kwargs):
    """
    Bridge function to allow Graph recommendation on new text.
    Ensure exactly n items are returned if possible.
    
    Requires kwargs:
    - model: SentenceTransformer model
    - embeddings: Path to embeddings of the dataset
    - articles: List of article IDs corresponding to embeddings
    - index: Faiss index of embeddings
    - communities: Path to communities.json
    """
    
    # 1. On récupère plus de candidats (ex: n*3) que nécessaire via les embeddings
    # pour être sûr d'avoir assez de points d'entrée si les clusters sont petits.
    bridge_result = retrieve_similar_articles(
        query=query_text,
        model=kwargs['model'],
        embeddings=kwargs['embeddings'],
        articles=kwargs['articles'],
        index=kwargs['index'],
        top_n=n * 3, 
        use_ann=True
    )
    
    if bridge_result.empty:
        return []
    
    recommendations = []
    seen_ids = set() # Pour éviter les doublons
    i = 0
    
    # 2. Boucle : tant qu'on n'a pas n recos ET qu'il reste des candidats bridge
    while len(recommendations) < n and i < len(bridge_result):
        
        # Combien d'articles nous manquent-ils pour atteindre n ?
        needed = n - len(recommendations)
        
        entry_node_id = bridge_result.iloc[i]['article_id']
        
        # On récupère jusqu'à 'needed' candidats dans ce cluster
        cluster_candidates = get_representative(
            member_id=entry_node_id,
            file_path=kwargs['communities'],
            top_n=needed
        )
        
        # Ajout des candidats en vérifiant les doublons
        for candidate in cluster_candidates:
            if candidate not in seen_ids:
                recommendations.append(candidate)
                seen_ids.add(candidate)
                
                # Si on a atteint n, on arrête tout de suite
                if len(recommendations) == n:
                    break
        
        i += 1
        
    
    # Fallback : Si après avoir parcouru tout le graphe on a moins de n articles
    # (cas très rare où le graphe est vide ou déconnecté), on complète avec
    # les résultats bruts des embeddings (bridge).
    if len(recommendations) < n:
        for idx, row in bridge_result.iterrows():
            cand = row['article_id']
            if cand not in seen_ids:
                recommendations.append(cand)
                seen_ids.add(cand)
            if len(recommendations) == n:
                break
                
    return recommendations

## 1.3 Execution & Tests
The following cells run the pipeline: loading data, building the graph, and testing the retrieval.

In [7]:
community_detection()

--- Creating graph from filtered JSON (in memory)... ---
Graph created: 214793 nodes, 1276385 edges.


In [8]:
file_path = f'data/processed/communities_{d_type}.json'  

with open(file_path, 'r') as infile:
    commu = json.load(infile)
print(len(commu))

article_name = {}
number_citation = {}
with open(FILTERED_JSON_PATH, 'r' ) as infile:
    nodes = json.load(infile)
for article in nodes['articles']:
    article_name[article['id']] = article['title']

36989


In [4]:
# Core Python libraries
import numpy as np
import pandas as pd
from tqdm import tqdm
import json
import re
import os

# NLP and Embeddings
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
import faiss


# Visualization and evaluation
import matplotlib.pyplot as plt
import seaborn as sns



/Users/surprisedcat/DTU/DS/DTU_DS_PROJECT_69/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [34]:
build_faiss_index()

<faiss.swigfaiss_avx2.IndexHNSWFlat; proxy of <Swig Object of type 'faiss::IndexHNSWFlat *' at 0x0000024AFC06BE70> >

---

# Part 2: Locality Sensitive Hashing (LSH)

This method uses probabilistic hashing to find similar documents quickly. It converts text to shingles, computes MinHash signatures, and uses Banding for candidate generation.

## 2.1 Libraries & Imports
Libraries for hashing (mmh3) and array manipulation.

In [11]:
import mmh3
import json 
from typing import Dict, Any
import numpy as np
from tqdm import tqdm
from mmh3 import hash

## 2.2 Function Definitions
Core LSH logic:
- `shingle`: Converts text to k-shingles.
- `minhash` & `signatures`: Creates compact signature matrices.
- `lsh_band_hash`: Implements the banding technique.
- `lsh`: Main retrieval function using Jaccard similarity.

In [12]:
def find_index(L, x):
    """
    Calculates the index (i) of the first occurrence of element x in list L.

    Args:
        L (list): The list to search within.
        x (any): The element whose index is being sought.

    Returns:
        int: The index of element x in L.

    Raises:
        ValueError: If element x is not found in list L.
    """
    try:
        # The index() method returns the index of the first occurrence
        # of the specified element.
        index = L.index(x)
        return index
    except ValueError:
        # index() raises a ValueError if the element is not found.
        # It's good practice to handle this error.
        raise ValueError(f"The element '{x}' is not in the list.")

In [13]:
punctuation = ['.',',',';',':','"']


def string_modulo_q(q, caracter_list, beginning_index):
    """takes a caracter list and a beginning index and returns a string composed of 
     the caracters in order beginning at the beginning index """
    assert q == len(caracter_list)

    string=""

    for i in range(q):
        string += caracter_list[(beginning_index+i)%q]

    return string


def shingle(q, text, punctuation_list = punctuation):
    "Returns the set of q-shingles from the original text"
    n = len(text)
    assert(n>q), "Text too short or shingle too long"
    assert q>=2, "Shingle size must be > 1 caracter "

    ind = 0 # index of the first caracter
    q_list = ["" for _ in range(q)]
    S = []
    ind_modified = 0

    for c in text:
        if (not (c in punctuation_list)) & (c != ' ') : # We ignore punctuation
            
            # Update q_string with another caracter
            q_list[ind] = c
            # Add one to the beginning index
            new_ind = (ind+1)%q
            ind = new_ind
            ind_modified += 1
            # New shingle added to the list
            S.append(string_modulo_q(q = q, caracter_list = q_list, beginning_index = ind))
    return S[q-1:]

In [14]:
# hashes a list of strings
def listhash(l,seed):
	val = 0
	for e in l:
		val = val ^ hash(e, seed)
	return val 

# Minhash function
def minhash(shingle_list, k):
    "returns a list of k different minhashes of the shingle list"
    return [min(listhash(s, seed) for s in shingle_list) for seed in range(k)]


In [27]:
def signatures(doc_list, shingle_size, signature_size):
    """
    inputs :
        - doc_lists : list of document of the form {'id': ... , 'abstract' : ... }
        - signature_size : size of the signatures
    outputs :
        - sig : signature matrix, every column represent the signature of a document
        - idx_to_sig : dictionnary that matches idexes (columns of sig) withs ids of the documents
    """
    idx_to_id = {} # dictionary of the signatures of each document
    n = len(doc_list)
    sig = np.zeros((signature_size,n))
    # signature of doc no "id" using minhashing on the shingle_list. Size of the shingles is shingle_size
    for i in range(n):
        id = doc_list[i]["id"]
        abstract = doc_list[i]["clean_text"]
        idx_to_id[i] = id
        sig[:,i] = np.array(minhash(shingle(q=shingle_size, text=abstract), k=signature_size))
    return sig, idx_to_id

In [16]:
def lsh_band_hash(band, m, lsh_seed) -> int:
    """
    Computes a hash value for a single band (a list of r integers).
    The goal is to map identical bands to the same hash bucket.

    band: A list of r integers representing the signature's portion 
            for a specific band.
    Returns: An integer hash value for the band.
    """
    
    band_string = ",".join(map(str, band)) # Convert the band to a string representation.
    
    hash_value = mmh3.hash(band_string, lsh_seed) # Compute the hash using MurmurHash3.
    
    return abs(hash_value) % m # Ensure non-negative and fit within m buckets

In [17]:
def Jaccard_similarity_signatures(input_signature, doc_signature) -> float :
    "Returns an approximation of the jaccard similarity between 2 documents doc_name1 and doc_name2 using signatures"
    sig = doc_signature 
    S = 0
    k = len(sig) # size of signature list
    assert k == len(input_signature), "Signatures are not matching size"
    for i in range(k): # Loop over the signature pairs from the two documents
        if sig[i]==input_signature[i]:
            S+=1
    return S/k

def Jaccard_similarity_shingles(shingles_list_A, shingles_list_B):
    """
    Calculates the Jaccard similarity coefficient between two lists of shingles.

    Args:
        shingles_list_A (list): The list of shingles for the first document.
        shingles_list_B (list): The list of shingles for the second document.

    Returns:
        float: The Jaccard similarity score (0.0 to 1.0).
    """

    # 1. Convert lists to sets for efficient set operations and to ensure uniqueness
    set_A = set(shingles_list_A)
    set_B = set(shingles_list_B)

    # 2. Calculate the size of the intersection (common shingles)
    intersection_size = len(set_A.intersection(set_B))
    
    # Alternatively: intersection_size = len(set_A & set_B)

    # 3. Calculate the size of the union (all unique shingles combined)
    union_size = len(set_A.union(set_B))
    
    # Alternatively: union_size = len(set_A | set_B)

    # 4. Calculate the Jaccard score
    if union_size == 0:
        # Avoid division by zero if both lists are empty
        return 0.0

    jaccard_score = intersection_size / union_size
    return jaccard_score


def Jaccard_similarity(input_text ,article_list, candidate_idx, q):
    return Jaccard_similarity_shingles(
        shingles_list_A= shingle(q = q, text = input_text),
        shingles_list_B= shingle(q = q, text = article_list[candidate_idx]["clean_text"]))

In [18]:
def Jaccard_similarity_signatures(input_signature, doc_signature) -> float :
    "Returns an approximation of the jaccard similarity between 2 documents doc_name1 and doc_name2 using signatures"
    sig = doc_signature 
    S = 0
    k = len(sig) # size of signature list
    assert k == len(input_signature), "Signatures are not matching size"
    for i in range(k): # Loop over the signature pairs from the two documents
        if sig[i]==input_signature[i]:
            S+=1
    return S/k

In [19]:
def Jaccard_similarity_shingles(shingles_list_A, shingles_list_B):
    """
    Calculates the Jaccard similarity coefficient between two lists of shingles.

    Args:
        shingles_list_A (list): The list of shingles for the first document.
        shingles_list_B (list): The list of shingles for the second document.

    Returns:
        float: The Jaccard similarity score (0.0 to 1.0).
    """

    # 1. Convert lists to sets for efficient set operations and to ensure uniqueness
    set_A = set(shingles_list_A)
    set_B = set(shingles_list_B)

    # 2. Calculate the size of the intersection (common shingles)
    intersection_size = len(set_A.intersection(set_B))
    
    # Alternatively: intersection_size = len(set_A & set_B)

    # 3. Calculate the size of the union (all unique shingles combined)
    union_size = len(set_A.union(set_B))
    
    # Alternatively: union_size = len(set_A | set_B)

    # 4. Calculate the Jaccard score
    if union_size == 0:
        # Avoid division by zero if both lists are empty
        return 0.0

    jaccard_score = intersection_size / union_size
    return jaccard_score


def Jaccard_similarity(input_text ,article_list, candidate_idx, q):
    return Jaccard_similarity_shingles(
        shingles_list_A= shingle(q = q, text = input_text),
        shingles_list_B= shingle(q = q, text = article_list[candidate_idx]["clean_text"]))

In [20]:
# Preprocessing the dataset to keep only the relevant information

def preprocess_lsh(dataset_path):
    """
    Input : 
        dataset_path : path of the json dataset of articles 
    Output :
        article_list : list of dictionnaries of the form {'id': ..., 'abstract': ...}  
    """
    try : 
        with open(dataset_path, 'r', encoding='utf-8') as f:
            data: Dict[str, Any] = json.load(f)
        print(f"Data succesfully loaded")
        
        article_list = [{'id': article['id'], 'clean_text' : article['clean_text']} for article in data] 
        return article_list

    #data = {'articles' : [d1 = {'id' : ..., 'authors' : ...,'abstract': ..., 'clean_text' : ... , 'categories' : ... , 'refs' :  ... } , d2, ...]}

    except Exception as e:
        print(f"Error in loading of the dataset : {e}")


In [21]:

# Implementing lsh function         

def lsh(input, article_list ,signature_matrix, idx_to_id, m, shingle_size, nb_band, band_size):
    """
    Inputs :
        input : input text from which we want to obtain sources
        article_list : list of arcticles i.e. dictionnaries of the format {'id': ..., 'abstract': ...}
        shingle_size : size of the shingle decomposition on which the minhashing is computed
        nb_band : number of horizontal bands in the signature matrix 
        band_size : number of rows per band in the signature matrix
        signature_size : size of the signatures of the documents obtained from minhashing of
                        the shingle size with signature_size different seeds. 
                        signature_size = nb_band*band_size

    Process :
        - Shingle all documents from the dataset and compute a signature for every document
          using minhashing (signatures function)
        - Find the documents that are most likely to be similar to input using LSH method
        - Compute the actual similarity between input and these document to eliminate false positives
    
    Outputs :
        - Most_similar : list of the most similar documents
        - Scores : list of Jaccard_similarities between input and documents 
    
    """

    # 1. Compute the a signature for every document

    k = band_size*nb_band # Signature size

    # Compute signature of input
    input_signature = signatures([{'id':'input', 'clean_text':input}],
                                 shingle_size = shingle_size,
                                 signature_size = k)[0][:,0]
    
    # 2. Find the documents that are most likely to be similar to input using LSH method

    similar_candidates = {}
    n = len(signature_matrix[0])  # number articles in the dataset

    for band_nb in range(nb_band):
        input_hash = lsh_band_hash(
            band = input_signature[band_nb*band_size:(band_nb+1)*band_size],
            m = m,
            lsh_seed = band_nb
        )
        for i in range(n):
            doc_hash = lsh_band_hash(
                band = signature_matrix[:,i][band_nb*band_size:(band_nb+1)*band_size],
                m = m,
                lsh_seed = band_nb
                )
            if doc_hash == input_hash :
                if i in similar_candidates :
                    similar_candidates[i] += 1
                else :
                    similar_candidates[i] = 1


    # 3. Compute the actual similarity between input and these documents


    Ordered_similar_candidates = similar_candidates.keys()
    Ordered_similarities = []
    for idx in similar_candidates :
        ## In case we want to compare the candidates on the jaccard similarities of the shingle lists
        #j = Jaccard_similarity(input_text= input, 
        #                       article_list= article_list,
        #                       candidate_idx=idx,
        #                       q=shingle_size)
        j = Jaccard_similarity_signatures(input_signature=input_signature, doc_signature=signature_matrix[:,idx])
        Ordered_similarities.append(j)
    Most_similar = zip(Ordered_similar_candidates,Ordered_similarities)
    Most_similar = sorted(Most_similar, key = lambda pair:pair[1], reverse = True)
    Scores = [ p[1] for p in Most_similar]
    Most_similar = [idx_to_id[p[0]]['id'] for p in Most_similar]
    
    return Most_similar, Scores


In [22]:
def lsh_n(n,input, article_list ,signature_matrix, idx_to_id, m, shingle_size, nb_band, band_size) : 
    """ Returns the Most_similar list of lsh with the n most relevant results only. If the number of result from lsh 
    is < n, random articles from the dataset are added"""

    Most_similar = lsh(input, article_list ,signature_matrix, idx_to_id, m, shingle_size, nb_band, band_size)[0]
    length = len(Most_similar)
    if length < n :
        c = 0
        while length < n :
            extra_id = article_list[c]['id']
            if extra_id not in Most_similar :
                Most_similar.append(extra_id)
                length += 1
            c += 1    
    return Most_similar[:n]

In [28]:
def calculate_top_n_accuracy(method_name, top_n_list, gold_set_path, **kwargs):
    """
    Calculates Top-N Accuracy for multiple N values simultaneously.
    
    Args:
        method_name (str): 'embedding', 'lsh', or 'graph'.
        top_n_list (list): List of integers for which to calculate accuracy (e.g. [1, 5, 10]).
        gold_set_path (str): Path to the .json gold set file.
        **kwargs: Variable arguments required for the specific methods.

    Returns:
        dict: A dictionary where keys are N and values are the accuracy scores.
    """
    
    # 1. Load Gold Set
    try:
        with open(gold_set_path, 'r', encoding='utf-8') as f:
            gold_set = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError) as e:
        print(f"Error loading gold set: {e}")
        return {}

    # 2. Determine the maximum K needed
    max_k = max(top_n_list)
    
    # Initialize counters for each k in the list
    hits_at_k = {k: 0 for k in top_n_list}
    
    # Pre-calculate total samples
    total_samples = 0
    for paper in gold_set:
        citations = paper.get('citations', {})
        for context_list in citations.values():
            total_samples += len(context_list)

    if total_samples == 0:
        return {k: 0.0 for k in top_n_list}

    print(f"--- Starting evaluation for {method_name} (Max K={max_k}) ---")
    pbar = tqdm(total=total_samples, desc="Processing Chunks")

    # 3. Main Loop
    for paper_entry in gold_set:
        citations = paper_entry.get('citations', {})
        
        for true_id, text_chunks in citations.items():
            for query_text in text_chunks:
                pbar.update(1)
                
                if not query_text or not query_text.strip():
                    continue

                # RETRIEVAL STEP
                recommended_ids = []
                
                try:
                    if method_name == "embedding":
                        df_results = retrieve_similar_articles(
                            query=query_text,
                            model=kwargs['model'],
                            embeddings=kwargs['embeddings'],
                            articles=kwargs['articles'],
                            index=kwargs['index'],
                            top_n=max_k,
                            use_ann=kwargs.get('use_ann', False)
                        )
                        recommended_ids = df_results['article_id'].tolist()

                    elif method_name == "LSH":
                        recommended_ids = lsh_n(
                            n=max_k,
                            input=query_text, 
                            article_list=kwargs['article_list'],
                            signature_matrix=kwargs['signature_matrix'],
                            idx_to_id=kwargs['idx_to_id'],
                            m=kwargs['m'],
                            shingle_size=kwargs['shingle_size'],
                            nb_band=kwargs['nb_band'],
                            band_size=kwargs['band_size'],
                        )

                    elif method_name == "graph":
                        recommended_ids = graph_retrieval_pipeline(
                            query_text=query_text,
                            n=max_k,
                            model=kwargs['model'],
                            embeddings=kwargs['embeddings'],
                            articles=kwargs['articles'],
                            index=kwargs['index'],
                            communities=kwargs['communities']
                        )
                    else:
                        raise ValueError(f"Unknown method: {method_name}")
                except Exception as e:
                    print(f"Error processing query: {e}") 
                    recommended_ids = []

                # METRIC CALCULATION STEP
                true_id_str = str(true_id)
                rec_ids_str = [str(x) for x in recommended_ids]
                
                # Check rank
                try:
                    rank = rec_ids_str.index(true_id_str)
                    position = rank + 1
                    
                    # Update counters:
                    for k in top_n_list:
                        if position <= k:
                            hits_at_k[k] += 1
                            
                except ValueError:
                    pass
    
    pbar.close()

    # 4. Final Calculation
    accuracies = {k: hits / total_samples for k, hits in hits_at_k.items()}
    
    return accuracies

## 2.3 Execution & Tests
Running the LSH pipeline: Preprocessing the dataset, computing signatures, and finding nearest neighbors.

In [ ]:
datatypes = ["most_cited", "quartiles", "stratified", "waterfilling"]
method = 2

In [23]:
json_path = f"data/clean_subdataset_{datatypes[method]}.json" 
data = preprocess_lsh(dataset_path = json_path)
print("preprocessing done")

Data succesfully loaded
preprocessing done


In [24]:
print(data[0])

{'id': '1404.3723', 'clean_text': 'we highlight the progress current status and open challenges of qcddriven\nphysics in theory and in experiment we discuss how the strong interaction is\nintimately connected to a broad sweep of physical problems in settings ranging\nfrom astrophysics and cosmology to stronglycoupled complex systems in\nparticle and condensedmatter physics as well as to searches for physics\nbeyond the standard model we also discuss how success in describing the strong\ninteraction impacts other fields and in turn how such subjects can impact\nstudies of the strong interaction in the course of the work we offer a\nperspective on the many research streams which flow into and out of qcd as\nwell as a vision for future developments'}


In [25]:
output_json_path = f'data/subdataset_lsh_{datatypes[method]}.json'
with open(output_json_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=4)

In [ ]:
data_path = f'data/subdataset_lsh_{datatypes[method]}.json'
with open(data_path, 'r', encoding='utf-8') as f:
            data: Dict[str, Any] = json.load(f)

: 

In [ ]:
q = 7 
b = 10
r = 10

signature_matrix_Nmost, idx_to_id = signatures(
    doc_list=data,
    shingle_size = q,
    signature_size = b*r
    )

Computing signatures:  17%|█▋        | 8529/49980 [1:49:58<1:20:07,  8.62it/s]      

In [ ]:
np.save(file = f"data/signature_lsh_{datatypes[method]}_q{q}_b{b}_r{r}", arr = signature_matrix_Nmost)
output_json_path = f"data/idx_to_id_lsh_{datatypes[method]}_q{q}_b{b}_r{r}.json"
with open(output_json_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=4)

In [26]:
data_path = f'data/subdataset_lsh_{datatypes[method]}.json'
with open(data_path, 'r', encoding='utf-8') as f:
            data: Dict[str, Any] = json.load(f)

q = 7 
b = 10
r = 10

try :
    signature_matrix = np.load(f"data/signature_lsh_{datatypes[method]}_q{q}_b{b}_r{r}.npy")
    with open(f"data/idx_to_id_lsh_{datatypes[method]}_q{q}_b{b}_r{r}.json", 'r', encoding='utf-8') as f:
            idx_to_id: Dict[str, Any] = json.load(f)
except FileNotFoundError as e :
    print(f"No signature matrix has been saved with the set of parameters : q = {q}, b = {b}, r = {r} ")

In [27]:
Most_similar_100, Scores = lsh(
        input = data[0]['clean_text'],
        article_list=data,
        signature_matrix = signature_matrix,
        idx_to_id = idx_to_id,
        shingle_size=q,
        m = signature_matrix.shape[1]//10, 
        nb_band = b,
        band_size = r,
        )

print("Most similar documents : " , Most_similar_100, '\n')
print("Scores : " , Scores)

Computing the signature of every document in the dataset ...
Computing signature of input ...


Computing signatures: 100%|██████████| 1/1 [00:00<00:00,  8.60it/s]


min hashing of the documents complete
Performing LSH to find similar candidates ...


LSH Bands: 100%|██████████| 10/10 [00:04<00:00,  2.02it/s]

LSH successfully performed to find similar candidates
Calculation of the actual similarities ... 

Most similar documents :  ['1404.3723', 'cond-mat/0402568', '1501.03060', '1407.4706', '1805.09127', '1811.06447', '1103.5690', '1011.6122', '1710.09302', '1310.5637', '1805.01853', 'math/0106271', '1401.6020', 'math/9811160', '0709.0668', '1704.01581', '1409.0356', '1802.05149', '1804.03579', '1307.8026', '1210.2058', '0908.1375', '1010.5456', '0710.0787', '1612.01216', '1809.11156', '1302.0134', 'math/0011227', '1505.03044', '1311.4290', '1004.2355', '1010.2065', '1609.03820', '1605.07473', '1603.00097', '1802.00844', '0802.3897', '1603.00710', '1604.06776', '1304.3772', '1211.0665', '1706.09869', '1812.03453', 'math/0202183', '1901.10231', 'cond-mat/0007400', '1812.08099', '1506.07540', '1802.04051', '1212.0133', '1508.02826', '0810.1622', '1604.08179', '0908.4353', '2104.05636', '1603.06914', '1807.09459', '1703.10505', '1707.04896', '1511.00367', '1612.03350', '1806.01110', '1310.574

In [28]:
n=10
Most_similar_100_n = lsh_n(
    n,
    input = data[0]['clean_text'],
        article_list=data,
        signature_matrix = signature_matrix,
        idx_to_id = idx_to_id,
        m = signature_matrix.shape[1]//10, 
        shingle_size = q,
        nb_band = b,
        band_size = r,
)

print(f"{n} most similar documents : ", Most_similar_100_n )

Computing the signature of every document in the dataset ...
Computing signature of input ...


Computing signatures: 100%|██████████| 1/1 [00:00<00:00, 11.21it/s]


min hashing of the documents complete
Performing LSH to find similar candidates ...


LSH Bands: 100%|██████████| 10/10 [00:04<00:00,  2.16it/s]

LSH successfully performed to find similar candidates
Calculation of the actual similarities ... 

10 most similar documents :  ['1404.3723', 'cond-mat/0402568', '1501.03060', '1407.4706', '1805.09127', '1811.06447', '1103.5690', '1011.6122', '1710.09302', '1310.5637']


---

# Part 3: Evaluation

Comparing the performance of Graph vs. LSH approaches using Recall@N metrics.

## 3.2 Evaluation Functions
Helper functions to calculate metrics and compare against gold standard datasets.

In [10]:
def common_commu(result_id, target_id):
    # are the result_id and the tardget_id in the same community / cluster ?
    COM_FILE = f'data/processed/communities_{d_type}.json'
    with open(COM_FILE, 'r') as file:
        commu = json.load(file)
    
    for commu_l in commu:
        if result_id in commu_l['community'] and target_id in commu_l['community']:
            return True
    
    return False

## 3.3 Results Visualization
Running the evaluation loops and plotting the results.

In [29]:
# --- Configuration ---
top_n_list = list(range(1, 21))
dataset_types = ["quartiles", "stratified"]

# Chemins statiques (globaux)
MODEL_NAME = 'all-MiniLM-L6-v2'

# --- Chargement des ressources globales (fait une seule fois) ---
print("Chargement du modèle SentenceTransformer et des Embeddings globaux...")
model = SentenceTransformer(MODEL_NAME)

# Initialisation du dictionnaire de résultats global
# Structure: global_results[type][method][n] = accuracy
global_results = {}

for d_type in dataset_types:
    print(f"\n{'='*60}")
    print(f"TRAITEMENT DU DATASET : {d_type.upper()}")
    print(f"{'='*60}")
    
    global_results[d_type] = {}
    
    # --- 1. Construction des chemins dynamiques ---
    gold_set_path = f"data/goldset/gold_dataset_cleaned_{d_type}.json"
    article_list_path = f"data/processed/clean_subdataset_{d_type}.json"
    
    # Chemins LSH
    sig_matrix_path = f"data/lsh/signature_lsh_{d_type}_q7_b10_r10.npy"
    idx_to_id_path = f"data/lsh/idx_to_id_lsh_{d_type}_q7_b10_r10.json"

    # Chemins Embeddings
    embeddings = f"data/embeddings/embeddings_{d_type}.npy"
    article_ids = f"data/embeddings/doc_ids_{d_type}.json"
    index_path = f"data/embeddings/faiss_{d_type}.index"

    # Chemins graph
    communities = f"data/processed/communities_{d_type}.json"
    
    # --- 2. Chargement des données spécifiques au dataset ---
    try:
        # Chargement de la liste d'articles (nécessaire pour mapper ID <-> Index)
        with open(article_list_path, 'r') as f:
            article_list_raw = json.load(f)
        article_list_data = article_list_raw['articles']

        # Chargement matrices LSH
        signature_matrix = np.load(sig_matrix_path)
        
        # Chargement idx_to_id
        with open(idx_to_id_path, 'r') as f:
            idx_to_id = json.load(f)

        # Parameters for lsh
        m = signature_matrix.shape[0]//10
        nb_band = 10
        band_size = 10

        # IDs for embedding
        article_ids_path = f"data/embeddings/doc_ids_{d_type}.json"
        with open(article_ids_path, 'r') as f:
            article_ids_list = json.load(f)
        
        # Faiss index for embedding
        current_faiss_index = faiss.read_index(index_path)
        print(f"Index FAISS chargé en mémoire pour {d_type}.")

        
    except Exception as e:
        print(f"ERREUR FATALE lors du chargement des données pour {d_type}: {e}")
        continue

    # --- 3. Évaluation des Méthodes ---
    
    # # A. Méthode Embedding
    # # --------------------
    # print(f"\n--- Évaluation Embedding ({d_type}) ---")
    # acc = calculate_top_n_accuracy(
    #     method_name="embedding",
    #     top_n_list=top_n_list,
    #     gold_set_path=gold_set_path,
    #     model=model,
    #     embeddings=embeddings,
    #     articles=article_ids_list,
    #     index=current_faiss_index,
    #     use_ann=True
    # )
    # global_results[d_type]['embedding'] = acc
    # # B. Méthode Graph (Hybrid)
    # # -------------------------
    # print(f"\n--- Évaluation Graph ({d_type}) ---")
    # acc = calculate_top_n_accuracy(
    #     method_name="graph",
    #     top_n_list=top_n_list,
    #     gold_set_path=gold_set_path,
    #     model=model,
    #     embeddings=embeddings,
    #     articles=article_ids_list,
    #     index=current_faiss_index,
    #     communities=communities
    # )
    # global_results[d_type]['graph'] = acc

    # C. Méthode LSH
    # --------------
    print(f"\n--- Évaluation LSH ({d_type}) ---")
    acc = calculate_top_n_accuracy(
        method_name="LSH",
        top_n_list=top_n_list,
        gold_set_path=gold_set_path,
        article_list=article_list_data,
        signature_matrix=signature_matrix,
        idx_to_id=idx_to_id,
        m=m,
        shingle_size=7,
        nb_band=nb_band,
        band_size=band_size
    )
    global_results[d_type]['LSH'] = acc

print("\nEvaluation Terminée !")
print("\n--- Résumé Accuracy @ 5 ---")
for dtype, methods in global_results.items():
    print(f"Dataset: {dtype}")
    for method, scores in methods.items():
        print(f"  - {method}: {scores.get(5, 0):.4f}")

# Save results
with open("evaluation_results_all_methods.json", "w") as f:
    json.dump(global_results, f, indent=4)

Chargement du modèle SentenceTransformer et des Embeddings globaux...

TRAITEMENT DU DATASET : QUARTILES
Index FAISS chargé en mémoire pour quartiles.

--- Évaluation LSH (quartiles) ---
--- Starting evaluation for LSH (Max K=20) ---












































































Processing Chunks:   0%|          | 22/7098 [04:31<24:17:56, 12.36s/it]

























































































































































































































































































































































































































































































































































































































































































































































































































































































TRAITEMENT DU DATASET : STRATIFIED
Index FAISS chargé en mémoire pour stratified.

--- Évaluation LSH (stratified) ---
--- Starting evaluation for LSH (Max K=20) ---


Processing Chunks: 100%|██████████| 5945/5945 [3:35:47<00:00,  2.18s/it]  


Evaluation Terminée !

--- Résumé Accuracy @ 5 ---
Dataset: quartiles
  - LSH: 0.0011
Dataset: stratified
  - LSH: 0.0013
